<a href="https://colab.research.google.com/github/Ghostalp07/EDA/blob/main/nodebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -U langgraph "langchain[anthropic]"


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.9/43.9 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.3/337.3 kB 16.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 16.5 MB/s eta 0:00:00


In [ ]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 41.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!pip install -qU langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 449.5/449.5 kB 17.3 MB/s eta 0:00:00


In [ ]:
!pip install -qU pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 9.9 MB/s eta 0:00:00


In [ ]:
!pip install langgraph langchain  faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 60.8 MB/s eta 0:00:00


In [ ]:
!pip install openai

loaders


In [ ]:
from langchain_community.document_loaders import PyPDFLoader


In [ ]:
pdfref = PyPDFLoader("/content/Numericals.pdf")
docs = pdfref.load()

In [ ]:
from openai import OpenAI
import os
import getpass
import os

if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

chunking


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
TextSplitter= RecursiveCharacterTextSplitter(chunk_size=1100,chunk_overlap=250)
chunks = TextSplitter.split_documents(docs)

langchain one


In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-large")
store = InMemoryVectorStore(embeddings)
store.add_documents(chunks)

In [ ]:
from langgraph.graph import StateGraph, END

In [ ]:
# class responser(dict):
#   ques:str
#   answer:str

# def fetcher(state:responser):
#   query = state["ques"]
#   docs = vectorstore.similarity_search(query, k=3)
#   return{"jawab: ".join([d.page_content for d in docs])}

# graph = StateGraph(responser)
# graph.add_node("retrieve", fetcher)
# graph.set_entry_point("retrieve")
# graph.set_finish_point("retrieve")

# app = graph.compile()

In [ ]:
from langchain_openai import OpenAI
def generate_with_source(query, context):
    prompt = f"""
    Answer the following question based on the provided context. If the answer comes from the document, mention that it was retrieved from the PDF. If the answer is generated by you (the LLM), mention that it comes from your internal knowledge base.

    Question: {query}

    Context: {context}


    Answer:
    """
    llm = OpenAI(model="gpt-3.5-turbo-instruct")

    response = llm.invoke(prompt)

    return response

In [ ]:
def jawab(query:int):
    results =store.similarity_search(query, k=3)
    context = "\n".join([f"{d.page_content} (Source: {d.metadata.get('source')})" for d in results])
    answer = generate_with_source(query, context)
    return answer


In [ ]:
query = "who is the prime minister of pakistan?"
answer = jawab(query)
print(answer)

In [ ]:
query = "Ironman 2 villain?"
answer = jawab(query)
print(answer)

Langraph

In [ ]:
from typing import TypedDict, Annotated, List
from langchain_core.documents import Document
import operator

class GraphState(TypedDict):
    question: str
    context: Annotated[List[Document], operator.add]
    answer: str

In [ ]:
from langchain_openai import OpenAI
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
def retrieve(state: GraphState):
    question = state["question"]
    results = store.similarity_search(question, k=3)
    state["context"] = results
    return state

def generate(state: GraphState):
    question = state["question"]
    context = "\n".join([f"{d.page_content} (Source: {d.metadata.get('source')})" for d in state["context"]])
    answer = generate_with_source(question, context)
    state["answer"] = answer
    return state

In [21]:


workflow = StateGraph(GraphState)

workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "generate")
workflow.set_finish_point("generate")
app = workflow.compile()

In [ ]:
app.invoke()
